# NB09 — Submission pipeline

**Standing rule (D-18):** this notebook must run end-to-end and produce both TSVs
passing `utils/validate_submission.py` after any edit anywhere in the project. A
change that breaks this path is not finished, however good the offline number.

**Stage S2** wires it up for the first time with the rule baseline (D-13):

1. Score the rule on the stratified holdout — establishes the local↔leaderboard
   calibration for every later stage.
2. Run the rule on the *test* sources and emit both `matching_results.tsv` and
   `candidate_pairs.tsv` (at S2 the candidate set == the matched set, per D-12).
3. Validate.

Uploading the file is a manual step — see the last cell.


## 1. Config cell


In [ ]:
# --- config cell: identical across notebooks ---
import os, sys, pathlib, time
_t_nb_start = time.time()
ON_SAGEMAKER = pathlib.Path("/opt/ml").exists() or "SM_CURRENT_HOST" in os.environ
ROOT = pathlib.Path(os.environ.get(
    "BER_ROOT",
    "/home/ec2-user/SageMaker/ber" if ON_SAGEMAKER
    else "/Users/pocketfm/Desktop/amazon_ML/Working Folder"
))
DATA, ART, OUT = ROOT/"student_resource/dataset", ROOT/"artifacts", ROOT/"output"
sys.path.insert(0, str(ROOT/"src"))
for p in (ART, OUT): p.mkdir(parents=True, exist_ok=True)
N_JOBS = os.cpu_count() or 1
SMOKE = bool(int(os.environ.get("BER_SMOKE", "0")))  # 1 = tiny slice for D-18
print(f"ON_SAGEMAKER={ON_SAGEMAKER}  ROOT={ROOT}  SMOKE={SMOKE}  N_JOBS={N_JOBS}")
# NOTE: this notebook is intended to run on SageMaker, not locally (D-24).


## 2. Score the rule baseline on the holdout

Uses the 149,999-row stratified holdout from S1. Predictions come from applying the
rule to `train_source*.tsv` restricted to holdout S1 entities.


In [ ]:
import time
import pandas as pd
from business_entity_resolution.rule_baseline import read_source, run_rule_baseline
from business_entity_resolution.metric import macro_f05, f_beta_per_entity

holdout = pd.read_parquet(ART/"holdout_s1.parquet")
print(f"holdout: {len(holdout):,} rows")

t = time.time()
train_s1 = read_source(DATA/"train"/"train_source1.tsv")
train_s2 = read_source(DATA/"train"/"train_source2.tsv")
train_s3 = read_source(DATA/"train"/"train_source3.tsv")
print(f"train sources loaded in {time.time()-t:.1f}s "
      f"(s1={len(train_s1):,} s2={len(train_s2):,} s3={len(train_s3):,})")

holdout_ids = set(holdout["entity_id"])
s1_slice = train_s1[train_s1["entity_id"].isin(holdout_ids)].reset_index(drop=True)

t = time.time()
preds_holdout = run_rule_baseline(s1_slice, train_s2, train_s3, exclusive=True)
print(f"rule baseline in {time.time()-t:.1f}s  "
      f"({len(preds_holdout):,} non-empty of {len(s1_slice):,} entities)")

truth = dict(zip(holdout["entity_id"], holdout["gt_matches"]))
holdout_score = macro_f05(preds_holdout, truth)
print(f"\n>>> rule baseline v0 macro F_0.5 on holdout: {holdout_score:.4f}")


### 2a. Per-country and per-cardinality breakdown


In [ ]:
from collections import defaultdict
per_country, per_bucket = defaultdict(list), defaultdict(list)
for eid, ctry, bucket, gt in zip(
    holdout.entity_id, holdout.country, holdout.cardinality_bucket, holdout.gt_matches
):
    s = f_beta_per_entity(preds_holdout.get(eid, ()), gt)
    per_country[ctry].append(s)
    per_bucket[int(bucket)].append(s)

print("by country:")
for c in sorted(per_country):
    xs = per_country[c]
    print(f"  {c:6s}  n={len(xs):>6,}  F_0.5={sum(xs)/len(xs):.4f}")
print("\nby cardinality bucket:")
for b in sorted(per_bucket):
    xs = per_bucket[b]
    print(f"  {b:>2}      n={len(xs):>6,}  F_0.5={sum(xs)/len(xs):.4f}")


## 3. Generate the submission on the *test* sources

Same rule, applied to the disjoint test files. In `SMOKE=1` mode only the first
~17k S1 entities are matched (the rest are emitted as empty rows) — used to
re-run the validator quickly after any code change per D-18.


In [ ]:
t = time.time()
test_s1 = read_source(DATA/"test"/"test_source1.tsv")
test_s2 = read_source(DATA/"test"/"test_source2.tsv")
test_s3 = read_source(DATA/"test"/"test_source3.tsv")
print(f"test sources loaded in {time.time()-t:.1f}s "
      f"(s1={len(test_s1):,} s2={len(test_s2):,} s3={len(test_s3):,})")

if SMOKE:
    test_s1_run = test_s1.head(len(test_s1)//100).reset_index(drop=True)
    print(f"SMOKE: matching first {len(test_s1_run):,} of {len(test_s1):,} S1 entities")
else:
    test_s1_run = test_s1

t = time.time()
preds_test = run_rule_baseline(test_s1_run, test_s2, test_s3, exclusive=True)
n_hit = sum(1 for v in preds_test.values() if v)
print(f"rule baseline in {time.time()-t:.1f}s  ({n_hit:,} non-empty)")


## 4. Emit both TSVs (matching + candidates) and validate


In [ ]:
from business_entity_resolution.config import detect_paths
from business_entity_resolution.submission import write_pair_of_tsvs, run_validator

paths = detect_paths()
# S2 cascade shape: matched == candidate (no later scorer yet — D-12)
n_m, n_c = write_pair_of_tsvs(paths, preds_test, preds_test)
print(f"wrote matching_rows={n_m:,} candidate_rows={n_c:,}")
for name in ("matching_results.tsv", "candidate_pairs.tsv"):
    p = paths.output/name
    with open(p) as f:
        head = [next(f) for _ in range(3)]
    print(f"\n--- {name} ({p.stat().st_size:,} bytes) ---")
    for line in head:
        print(repr(line))

rc, out = run_validator(paths)
print(out)
assert rc == 0, f"validator failed with exit {rc}"
print("D-18: submission path green.")


## 5. Log the run (D-19)


In [ ]:
import json, time
row = {
    "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "stage": "S2",
    "notebook": "NB09_submission",
    "config": {
        "strategy": "rule_baseline_organizer_address_abbrev",
        "keys": ["country", "name_key(drop_organizer_suffixes)", "address_key(rd_to_road_st_to_street)"],
        "exclusive": True,
        "smoke": SMOKE,
    },
    "metrics": {
        "holdout_macro_f05": round(float(holdout_score), 6),
        "holdout_by_country": {c: round(sum(xs)/len(xs), 6) for c, xs in per_country.items()},
        "test_predictions_nonempty": int(n_hit),
        "test_predictions_total_rows": int(n_m),
        "validator_exit": 0,
    },
    "notes": "S2 rule baseline. Local score is our calibration anchor — record the leaderboard offset once uploaded.",
}
with open(ART/"runs.jsonl", "a") as f:
    f.write(json.dumps(row) + "\n")
print(json.dumps(row, indent=2))


## 6. Paste-back block

The cell below prints one JSON blob and the tail of the validator output. Copy that
entire block back into the local chat — that is what gets appended to the local
`DECISIONS.md` and `artifacts/runs.jsonl`. No file needs to be synced back down
manually; the JSON carries everything.


In [ ]:
print("=========================== PASTE-BACK-START ===========================")
print(json.dumps(row, indent=2))
print("---")
print("wall_clock_seconds :", int(time.time() - _t_nb_start) if "_t_nb_start" in dir() else "n/a")
print("output_files:")
for name in ("matching_results.tsv", "candidate_pairs.tsv"):
    p = paths.output / name
    print(f"  {name}  bytes={p.stat().st_size:,}  mtime={time.strftime('%Y-%m-%dT%H:%M:%S', time.localtime(p.stat().st_mtime))}")
print("---")
print("validator tail:")
for line in out.strip().splitlines()[-6:]:
    print(f"  {line}")
print("============================ PASTE-BACK-END ============================")


## 7. Upload

Once the paste-back is captured, download `output/matching_results.tsv` from the
SageMaker file browser (right-click → *Download*) and upload it to the challenge
portal. Paste the returned leaderboard score back to the chat as well — that gives
us the holdout↔leaderboard offset D-03 depends on.

## Next

S2 code lands here; the *stage* is complete only once the paste-back is received.
After that, next stage is **S3 — mine the noise model → normalization v1**
(`NB03_normalization.ipynb`).
